In [46]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
import warnings

# Suppress routine warnings for sufficient execution output
warnings.filterwarnings(
    "ignore", category=UserWarning, module="joblib.externals.loky"
)
warnings.filterwarnings(
    "ignore", category=UserWarning, module="sklearn.metrics"
)


In [47]:
data_orig = pd.read_csv('system_Original.csv')
data_x = pd.read_csv('system_X.csv')
data_y = pd.read_csv('system_Y.csv')

# Tag each dataframe with it's respective source
data_orig['system'] = 'System_Original'
data_x['system'] = 'System_X'
data_y['system'] = 'System_Y'

# Concatenate into a unified dataframe using pd.concat
data = pd.concat([data_orig, data_x, data_y], ignore_index=True)

In [48]:
# Encode text labels into integer arrays for machine learning models
encoder = LabelEncoder()

y_human = encoder.fit_transform(data['target_human'])
y_asv = encoder.fit_transform(data['target_asv'])

In [49]:
# Extract speaker IDs for speaker-disjoint splitting
sid = data['sid'].to_numpy()

In [50]:
# Isolate feature matrix X by dropping non-feature metadata columns (ended up dropping 'filename' as well as this column distorted the results)
X = data.drop(columns=['filename', 'sid', 'uid', 'gender', 'target_human', 'target_asv', 'system']).to_numpy()
print('Features:', X.shape, ' Labels:', y_human.shape, y_asv.shape)

Features: (4923, 88)  Labels: (4923,) (4923,)


In [51]:
# Use GroupShuffleSplit based on speaker ID (sid) to guarantee speaker independence
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_val_human, test_human = next(gss.split(X, y_human, sid))
train_val_asv, test_asv = next(gss.split(X, y_asv, sid))

X_train_val_human, X_test_human = X[train_val_human], X[test_human]
y_train_val_human, y_test_human = y_human[train_val_human], y_human[test_human]

X_train_val_asv, X_test_asv = X[train_val_asv], X[test_asv]
y_train_val_asv, y_test_asv = y_asv[train_val_asv], y_asv[test_asv]

sid_train_val_human = sid[train_val_human]
sid_test_human = sid[test_human]

sid_train_val_asv = sid[train_val_asv]
sid_test_asv = sid[test_asv]

print(f"Human Train/Val speakers count: {len(np.unique(sid_train_val_human))}")
print(f"ASV Train/Val speakers count: {len(np.unique(sid_train_val_asv))}")
print(f"Human Test speakers count: {len(np.unique(sid_test_human))}")
print(f"ASV Test speakers count: {len(np.unique(sid_test_asv))}")

def get_label_counts(y_arr, i_train_val, i_test, encoder):
    train_val_labels = y_arr[i_train_val]
    test_labels = y_arr[i_test]
    classes = encoder.classes_
    res = []
    for i, c in enumerate(classes):
        train_val_count = np.sum(train_val_labels == i)
        test_count = np.sum(test_labels == i)
        res.append({'Class': c, 'Train/Val Count': train_val_count, 'Test Count': test_count})
    return pd.DataFrame(res)

print("\nHuman Labels:")
print(get_label_counts(y_human, train_val_human, test_human, encoder))
print("\nASV Labels:")
print(get_label_counts(y_asv, train_val_asv, test_asv, encoder))

Human Train/Val speakers count: 24
ASV Train/Val speakers count: 24
Human Test speakers count: 6
ASV Test speakers count: 6

Human Labels:
   Class  Train/Val Count  Test Count
0   goat              872         161
1   lamb              261          58
2  sheep             2270         651
3   wolf              539         111

ASV Labels:
   Class  Train/Val Count  Test Count
0   goat              507         221
1   lamb              154           0
2  sheep             2986         760
3   wolf              295           0


In [52]:
# Setup Stratified Group K-Fold cross-validation (5 folds) to respect speaker groups and class ratios
sgkf = StratifiedGroupKFold(n_splits=5)

for fold, (train, val) in enumerate(sgkf.split(X_train_val_human, y_train_val_human, sid_train_val_human)):
    X_train_human, X_val_human = X_train_val_human[train], X_train_val_human[val]
    y_train_human, y_val_human = y_train_val_human[train], y_train_val_human[val]
    
    print(f" Fold {fold}: Train speakers = {len(np.unique(sid_train_val_human[train]))}, Val speakers = {len(np.unique(sid_train_val_human[val]))}")

for fold, (train, val) in enumerate(sgkf.split(X_train_val_asv, y_train_val_asv, sid_train_val_asv)):
    X_train_asv, X_val_asv = X_train_val_asv[train], X_train_val_asv[val]
    y_train_asv, y_val_asv = y_train_val_asv[train], y_train_val_asv[val]
    
    print(f" Fold {fold}: Train speakers = {len(np.unique(sid_train_val_asv[train]))}, Val speakers = {len(np.unique(sid_train_val_asv[val]))}")

 Fold 0: Train speakers = 19, Val speakers = 5
 Fold 1: Train speakers = 20, Val speakers = 4
 Fold 2: Train speakers = 19, Val speakers = 5
 Fold 3: Train speakers = 19, Val speakers = 5
 Fold 4: Train speakers = 19, Val speakers = 5
 Fold 0: Train speakers = 20, Val speakers = 4
 Fold 1: Train speakers = 19, Val speakers = 5
 Fold 2: Train speakers = 19, Val speakers = 5
 Fold 3: Train speakers = 19, Val speakers = 5
 Fold 4: Train speakers = 19, Val speakers = 5


In [53]:
# Standardise features using StandardScaler to fit only on training/validation splits
scaler_human = StandardScaler()
X_tv_human_scaled = scaler_human.fit_transform(X_train_val_human)
X_test_human_scaled = scaler_human.transform(X_test_human)

scaler_asv = StandardScaler()
X_tv_asv_scaled = scaler_asv.fit_transform(X_train_val_asv)
X_test_asv_scaled = scaler_asv.transform(X_test_asv)

In [54]:
# Check for missing values across datasets
print("Missing values across datasets:")
print(f"Original: {data_orig.isnull().sum().sum()}, X: {data_x.isnull().sum().sum()}, Y: {data_y.isnull().sum().sum()}")

Missing values across datasets:
Original: 0, X: 0, Y: 0


In [55]:
print("Target Human Class Balance")
print(data['target_human'].value_counts())

print("\nTarget ASV Class Balance ")
print(data['target_asv'].value_counts())

Target Human Class Balance
target_human
sheep    2921
goat     1033
wolf      650
lamb      319
Name: count, dtype: int64

Target ASV Class Balance 
target_asv
sheep    3746
goat      728
wolf      295
lamb      154
Name: count, dtype: int64


In [56]:
knn_tuning_results_human = []
# Define hyperparameter grid: neighbors (k) and distance weighting schemes
for k in [3, 5, 7, 11]:
    for weights in ['uniform', 'distance']:
        scores = []
        # Iterate over speaker-disjoint, stratified cross-validation folds
        for train, val in sgkf.split(X_tv_human_scaled, y_human[train_val_human], sid[train_val_human]):
            knn = KNeighborsClassifier(n_neighbors=k, weights=weights)
            knn.fit(X_tv_human_scaled[train], y_human[train_val_human][train])
            preds = knn.predict(X_tv_human_scaled[val])
            scores.append(f1_score(y_human[train_val_human][val], preds, average='macro', zero_division=0))
        knn_tuning_results_human.append({
            'Algorithm': 'KNN',
            'Hyperparameters': f'n_neighbors={k}, weights={weights}',
            'Mean Validation Macro F1': np.mean(scores),
            'Std Validation Macro F1': np.std(scores)
        })

knn_tuning_results_asv = []
for k in [3, 5, 7, 11]:
    for weights in ['uniform', 'distance']:
        scores = []
        for train, val in sgkf.split(X_tv_asv_scaled, y_asv[train_val_asv], sid[train_val_asv]):
            knn = KNeighborsClassifier(n_neighbors=k, weights=weights)
            knn.fit(X_tv_asv_scaled[train], y_asv[train_val_asv][train])
            preds = knn.predict(X_tv_asv_scaled[val])
            scores.append(f1_score(y_asv[train_val_asv][val], preds, average='macro', zero_division=0))
        knn_tuning_results_asv.append({
            'Algorithm': 'KNN',
            'Hyperparameters': f'n_neighbors={k}, weights={weights}',
            'Mean Validation Macro F1': np.mean(scores),
            'Std Validation Macro F1': np.std(scores)
        })


In [57]:
rf_tuning_results_human = []
# Define hyperparameter grid: number of estimators and maximum tree depth
for n_est in [50, 100, 200]:
    for max_d in [None, 10, 20]:
        scores = []
        for train, val in sgkf.split(X_tv_human_scaled, y_human[train_val_human], sid[train_val_human]):
            rf = RandomForestClassifier(n_estimators=n_est, max_depth=max_d, random_state=42)
            rf.fit(X_tv_human_scaled[train], y_human[train_val_human][train])
            preds = rf.predict(X_tv_human_scaled[val])
            scores.append(f1_score(y_human[train_val_human][val], preds, average='macro', zero_division=0))
        rf_tuning_results_human.append({
            'Algorithm': 'Random Forest',
            'Hyperparameters': f'n_estimators={n_est}, max_depth={max_d}',
            'Mean Validation Macro F1': np.mean(scores),
            'Std Validation Macro F1': np.std(scores)
        })

rf_tuning_results_asv = []
for n_est in [50, 100, 200]:
    for max_d in [None, 10, 20]:
        scores = []
        for train, val in sgkf.split(X_tv_asv_scaled, y_asv[train_val_asv], sid[train_val_asv]):
            # Set random_state=42 for deterministic tree induction and reproducibility (I still don't understand why 42 is the meaning of life)
            rf = RandomForestClassifier(n_estimators=n_est, max_depth=max_d, random_state=42)
            rf.fit(X_tv_asv_scaled[train], y_asv[train_val_asv][train])
            preds = rf.predict(X_tv_asv_scaled[val])
            scores.append(f1_score(y_asv[train_val_asv][val], preds, average='macro', zero_division=0))
        rf_tuning_results_asv.append({
            'Algorithm': 'Random Forest',
            'Hyperparameters': f'n_estimators={n_est}, max_depth={max_d}',
            'Mean Validation Macro F1': np.mean(scores),
            'Std Validation Macro F1': np.std(scores)
        })

In [58]:
df_tuning_human = pd.DataFrame(knn_tuning_results_human + rf_tuning_results_human)
df_tuning_asv = pd.DataFrame(knn_tuning_results_asv + rf_tuning_results_asv)
print(df_tuning_human.to_string())

print(df_tuning_asv.to_string())

        Algorithm                   Hyperparameters  Mean Validation Macro F1  Std Validation Macro F1
0             KNN    n_neighbors=3, weights=uniform                  0.231807                 0.027335
1             KNN   n_neighbors=3, weights=distance                  0.235347                 0.028941
2             KNN    n_neighbors=5, weights=uniform                  0.236488                 0.021179
3             KNN   n_neighbors=5, weights=distance                  0.240589                 0.029486
4             KNN    n_neighbors=7, weights=uniform                  0.231967                 0.026034
5             KNN   n_neighbors=7, weights=distance                  0.233745                 0.030260
6             KNN   n_neighbors=11, weights=uniform                  0.232536                 0.029490
7             KNN  n_neighbors=11, weights=distance                  0.233448                 0.032324
8   Random Forest   n_estimators=50, max_depth=None                  0.25

In [59]:
# Tune KNN hyperparameters using inner cross-validation on train/val data.
def tune_evaluate_knn(X_tv, y_tv, sid_tv):
    best_score = -1
    best_params = {'n_neighbors': 5, 'weights': 'uniform'}
    for k in [3, 5, 7, 11]:
        for weights in ['uniform', 'distance']:
            scores = []
            for train, val in sgkf.split(X_tv, y_tv, sid_tv):
                knn = KNeighborsClassifier(n_neighbors=k, weights=weights)
                knn.fit(X_tv[train], y_tv[train])
                y_val_pred = knn.predict(X_tv[val])
                scores.append(f1_score(y_tv[val], y_val_pred, average='macro', zero_division=0))
            mean_score = np.mean(scores)
            if mean_score > best_score:
                best_score = mean_score
                best_params = {'n_neighbors': k, 'weights': weights}

    best_knn = KNeighborsClassifier(**best_params)
    best_knn.fit(X_tv, y_tv)
    return best_knn, best_params

In [60]:
# Tune Random Forest hyperparameters using inner cross-validation on train/val data.
def tune_evaluate_rf(X_tv, y_tv, sid_tv):
    best_score = -1
    best_params = {'n_estimators': 100, 'max_depth': None}
    for n_est in [50, 100]:
        for max_d in [None, 10]:
            scores = []
            for train, val in sgkf.split(X_tv, y_tv, sid_tv):
                rf = RandomForestClassifier(n_estimators=n_est, max_depth=max_d, random_state=42, n_jobs=-1)
                rf.fit(X_tv[train], y_tv[train])
                y_val_pred = rf.predict(X_tv[val])
                scores.append(f1_score(y_tv[val], y_val_pred, average='macro', zero_division=0))
            mean_score = np.mean(scores)
            if mean_score > best_score:
                best_score = mean_score
                best_params = {'n_estimators': n_est, 'max_depth': max_d}
                
    best_rf = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
    best_rf.fit(X_tv, y_tv)
    return best_rf, best_params

In [61]:
systems = ['System_Original', 'System_X', 'System_Y', 'Combined']
experiment_results = []

names = [
    {
        'label_name': 'Human',
        'y_tv': y_train_val_human,
        'y_te': y_test_human,
        'sid_tv': sid_train_val_human,
        'sid_te': sid_test_human,
        'X_tv_scaled': X_tv_human_scaled,
        'X_te_scaled': X_test_human_scaled,
        'tv_arr': train_val_human,
        'te_arr': test_human
    },
    {
        'label_name': 'ASV',
        'y_tv': y_train_val_asv,
        'y_te': y_test_asv,
        'sid_tv': sid_train_val_asv,
        'sid_te': sid_test_asv,
        'X_tv_scaled': X_tv_asv_scaled,
        'X_te_scaled': X_test_asv_scaled,
        'tv_arr': train_val_asv,
        'te_arr': test_asv
    }
]

# Training and evaluation loop across all systems and labels
for n in names:
    label = n['label_name']
    y_tv, y_te = n['y_tv'], n['y_te']
    sid_tv, sid_te = n['sid_tv'], n['sid_te']
    X_tv_scaled, X_te_scaled = n['X_tv_scaled'], n['X_te_scaled']
    tv_arr, te_arr = n['tv_arr'], n['te_arr']
    
    sys_col_tv = data['system'].to_numpy()[tv_arr]
    sys_col_te = data['system'].to_numpy()[te_arr]
    
    for sys_name in systems:
        if sys_name == 'Combined':
            mask_tv = np.ones(len(X_tv_scaled), dtype=bool)
            mask_te = np.ones(len(X_te_scaled), dtype=bool)
        else:
            mask_tv = (sys_col_tv == sys_name)
            mask_te = (sys_col_te == sys_name)
            
        X_tr_sys = X_tv_scaled[mask_tv]
        y_tr_sys = y_tv[mask_tv]
        sid_tr_sys = sid_tv[mask_tv]
        
        X_te_sys = X_te_scaled[mask_te]
        y_te_sys = y_te[mask_te]

        # Tune and train models on system-specific training data
        knn_model, _ = tune_evaluate_knn(X_tr_sys, y_tr_sys, sid_tr_sys)
        rf_model, _ = tune_evaluate_rf(X_tr_sys, y_tr_sys, sid_tr_sys)

        # Evaluate models on held out system test data
        for algorithm, model in [('KNN', knn_model), ('RandomForest', rf_model)]:
            y_pred = model.predict(X_te_sys)
            experiment_results.append({
                'System': sys_name,
                'Label': label,
                'Algorithm': algorithm,
                'Accuracy': accuracy_score(y_te_sys, y_pred),
                'Macro_F1': f1_score(y_te_sys, y_pred, average='macro', zero_division=0),
                'Weighted_F1': f1_score(y_te_sys, y_pred, average='weighted', zero_division=0),
                'Balanced_Acc': balanced_accuracy_score(y_te_sys, y_pred)
            })

results_df = pd.DataFrame(experiment_results)
print(results_df.to_string())

             System  Label     Algorithm  Accuracy  Macro_F1  Weighted_F1  Balanced_Acc
0   System_Original  Human           KNN  0.357798  0.153244     0.393636      0.275213
1   System_Original  Human  RandomForest  0.385321  0.140312     0.370732      0.291667
2          System_X  Human           KNN  0.422018  0.260228     0.429522      0.333119
3          System_X  Human  RandomForest  0.373089  0.228173     0.377800      0.295450
4          System_Y  Human           KNN  0.737003  0.282864     0.708457      0.441392
5          System_Y  Human  RandomForest  0.666667  0.400000     0.667890      0.399267
6          Combined  Human           KNN  0.512742  0.252250     0.502461      0.258278
7          Combined  Human  RandomForest  0.503568  0.202321     0.464868      0.218927
8   System_Original    ASV           KNN  0.972477  0.328682     0.986047      0.972477
9   System_Original    ASV  RandomForest  1.000000  1.000000     1.000000      1.000000
10         System_X    ASV      